# CGA Ray Tracer using Kingdon & Elements

**Based on Chapter 23 of Dorst, Fontijne & Mann — *Geometric Algebra for Computer Science* (Morgan Kaufmann, 2010)**

This notebook implements a complete ray tracer using Conformal Geometric Algebra (CGA) via the [Kingdon](https://github.com/tBuLi/kingdon) Python library. We follow the chapter's structure:

| Section | Topic |
|---------|-------|
| 23.1 | Ray-tracing basics |
| 23.3 | Representing meshes — vertices, faces, bounding spheres |
| 23.4 | Modeling the scene — rotors for translation, rotation, scaling |
| 23.5 | Tracing the rays — representation, spawning, intersection, reflection, refraction |
| 23.6 | Shading — ambient, diffuse, specular, shadows |

Key conventions (matching the book):
- `no` / `eo` = origin null vector $e_o$
- `ni` / `ei` = infinity null vector $e_\infty$
- `<<` in the book = inner product (left contraction) = `|` in Kingdon
- `^` = outer (wedge) product
- `*` = geometric product

In [ ]:
# Cell 1 — Imports and CGA algebra setup
import numpy as np
import math
from kingdon import Algebra
from PIL import Image
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, List, Tuple
import time

# ---------- CGA Cl(4,1) ----------
cga = Algebra(4, 1)

# Euclidean basis
e1 = cga.blades['e1']
e2 = cga.blades['e2']
e3 = cga.blades['e3']

# Extra dimensions
e4 = cga.blades['e4']   # e4^2 = +1
e5 = cga.blades['e5']   # e5^2 = -1

# Null basis (Dorst convention: no = origin, ni = infinity)
no = 0.5 * (e5 - e4)    # origin
ni = e4 + e5             # infinity

# Pseudoscalar
I5 = e1 * e2 * e3 * e4 * e5   # I5^2 = -1  =>  I5^{-1} = -I5

# Scalar blade for rotor construction
ONE = cga.blades['e']    # the scalar unit

print(f'CGA Cl(4,1) ready  |  no·ni = {no | ni}  |  I5² = {I5 * I5}')

## CGA Utility Functions

Following Section 23.3 — we define the fundamental CGA primitives:
- **`cgaPoint`** — embed a 3-D Euclidean point as a normalized conformal point $P = e_o + \mathbf{x} + \frac{1}{2}|\mathbf{x}|^2 e_\infty$
- **`down`** — extract Euclidean coordinates back from a conformal point
- **`down_flat`** — extract coordinates from a flat point (result of intersections)
- Dual / undual helpers

In [ ]:
# Cell 2 — CGA primitive helpers

def scalar_val(mv):
    """Extract the scalar (grade-0) part of a multivector as a float."""
    items = list(mv.items())
    if len(items) > 0 and items[0][0] == 0:
        return float(items[0][1])
    return 0.0


def get_coeff(mv, blade_name):
    """Get coefficient of a named blade from a multivector."""
    blade = cga.blades[blade_name]
    blade_items = dict(zip(blade.keys(), blade.values()))
    mv_items = dict(zip(mv.keys(), mv.values()))
    for k, v in blade_items.items():
        if k in mv_items:
            return float(mv_items[k]) / float(v)
    return 0.0


def cgaPoint(x, y, z):
    """Normalized conformal point  P = no + x + 0.5*|x|^2 * ni  (eq 13.3)."""
    return no + x * e1 + y * e2 + z * e3 + 0.5 * (x*x + y*y + z*z) * ni


def cgaPointVec(v):
    """cgaPoint from a numpy [x,y,z] array."""
    return cgaPoint(float(v[0]), float(v[1]), float(v[2]))


def down(P):
    """Extract Euclidean [x,y,z] from a normalized CGA point (grade 1)."""
    w = scalar_val(-(P | ni))
    if abs(w) < 1e-12:
        return np.zeros(3)
    x = scalar_val(P | e1) / w
    y = scalar_val(P | e2) / w
    z = scalar_val(P | e3) / w
    return np.array([x, y, z])


def down_flat(fp):
    """Extract Euclidean [x,y,z] from a CGA flat point (grade 2).
    A flat point is  p ^ ni  with basis components  ei^e4, ei^e5, e4^e5."""
    w = -get_coeff(fp, 'e45')
    if abs(w) < 1e-12:
        return None
    x = get_coeff(fp, 'e14') / w
    y = get_coeff(fp, 'e24') / w
    z = get_coeff(fp, 'e34') / w
    return np.array([x, y, z])


def cga_dual(X):
    """CGA dual:  X* = X · I5^{-1} = X · (-I5) = -X·I5."""
    return X * (-I5)


def cga_undual(X):
    """CGA undual:  X = X* · I5."""
    return X * I5


# ---- quick tests ----
P = cgaPoint(1, 2, 3)
assert np.allclose(down(P), [1, 2, 3]), 'down∘cgaPoint roundtrip failed'

fp = P ^ ni
assert np.allclose(down_flat(fp), [1, 2, 3]), 'down_flat failed'

print('CGA utilities OK:', down(P), down_flat(fp))

## Section 23.3 — Representing Meshes

We define `Vertex`, `Face`, and `Mesh` classes following the book.

- Each **vertex** stores a CGA *normalized point*, a *surface attitude* (free bivector = dual of normal), and texture coordinates.
- Each **face** stores vertex indices, the CGA *plane* through the three vertices, and the three *edge lines*.
- Each **mesh** computes a CGA *bounding sphere*.

In [ ]:
# Cell 3 — Mesh representation (Section 23.3)

@dataclass
class Vertex:
    """CGA vertex: normalized point + surface attitude + tex coords."""
    pos: np.ndarray                # Euclidean [x,y,z]  (kept for fast access)
    pt: object = None              # CGA normalized point
    normal: np.ndarray = None      # Euclidean normal
    tex: np.ndarray = field(default_factory=lambda: np.zeros(2))
    
    def __post_init__(self):
        self.pt = cgaPointVec(self.pos)
        if self.normal is None:
            self.normal = np.array([0., 1., 0.])


@dataclass
class Face:
    """Triangular face: three vertex indices + precomputed CGA blades."""
    idx: Tuple[int, int, int]
    plane_cga: object = None       # CGA plane  v0^v1^v2^ni
    edge_lines: list = field(default_factory=list)
    normal: np.ndarray = None      # Euclidean face normal


class Mesh:
    """Polygonal mesh with CGA bounding sphere (Section 23.3)."""
    
    def __init__(self, vertices: List[Vertex], faces_idx: List[Tuple[int,int,int]],
                 color=np.array([0.8, 0.8, 0.8]),
                 reflectivity=0.0, transparency=0.0, ior=1.0):
        self.vertices = vertices
        self.faces: List[Face] = []
        self.color = np.asarray(color, dtype=float)
        self.reflectivity = reflectivity
        self.transparency = transparency
        self.ior = ior  # index of refraction
        
        # Build faces with CGA data
        for idx in faces_idx:
            self._build_face(idx)
        
        # Bounding sphere (Section 23.3)
        self.bounding_center, self.bounding_radius = self._compute_bounding_sphere()
        self.bounding_sphere_dual = cgaPointVec(self.bounding_center) \
                                    - 0.5 * self.bounding_radius**2 * ni
    
    # ------------------------------------------------------------------
    def _build_face(self, idx):
        v0, v1, v2 = [self.vertices[i] for i in idx]
        
        # CGA plane: v0.pt ^ v1.pt ^ v2.pt ^ ni  (book p.563)
        pl = v0.pt ^ v1.pt ^ v2.pt ^ ni
        
        # Edge lines  (book p.563)
        el = [
            v0.pt ^ v1.pt ^ ni,
            v1.pt ^ v2.pt ^ ni,
            v2.pt ^ v0.pt ^ ni,
        ]
        
        # Euclidean face normal from vertex positions
        e01 = v1.pos - v0.pos
        e02 = v2.pos - v0.pos
        n = np.cross(e01, e02)
        nn = np.linalg.norm(n)
        if nn > 1e-12:
            n = n / nn
        
        self.faces.append(Face(idx=idx, plane_cga=pl, edge_lines=el, normal=n))
    
    # ------------------------------------------------------------------
    def _compute_bounding_sphere(self):
        """Bounding sphere — Section 23.3 (book p.563-564).
        Center at midpoint of axis-aligned extent; radius = max distance."""
        pts = np.array([v.pos for v in self.vertices])
        lo = pts.min(axis=0)
        hi = pts.max(axis=0)
        center = 0.5 * (lo + hi)
        radius = np.max(np.linalg.norm(pts - center, axis=1)) * 1.01  # +1% margin
        return center, radius


# --- quick test: a single triangle ---
verts = [
    Vertex(pos=np.array([0., 0., 0.]), normal=np.array([0., 1., 0.])),
    Vertex(pos=np.array([1., 0., 0.]), normal=np.array([0., 1., 0.])),
    Vertex(pos=np.array([0., 0., 1.]), normal=np.array([0., 1., 0.])),
]
mesh_test = Mesh(verts, [(0, 1, 2)])
print(f'Mesh: {len(mesh_test.faces)} face(s), '
      f'bounding sphere center={mesh_test.bounding_center}, r={mesh_test.bounding_radius:.3f}')
print(f'Face normal: {mesh_test.faces[0].normal}')

## Scene Primitives — Analytic Sphere & Plane

For the ray tracer we also support **analytic** spheres and infinite planes (not just triangle meshes). These are naturally represented in CGA:

| Primitive | IPNS (dual) representation |
|-----------|---------------------------|
| Sphere | $\hat{S} = P - \frac{r^2}{2} e_\infty$ |
| Plane | $\hat{\pi} = \mathbf{n} + d\, e_\infty$ where $\mathbf{n}\cdot\mathbf{x} = d$ |

In [ ]:
# Cell 4 — Scene primitives and CGA transformations (Section 23.4)

@dataclass
class Material:
    color: np.ndarray = field(default_factory=lambda: np.array([0.8, 0.8, 0.8]))
    ambient: float = 0.1
    diffuse: float = 0.7
    specular: float = 0.5
    shininess: float = 32.0
    reflectivity: float = 0.0
    transparency: float = 0.0
    ior: float = 1.5    # index of refraction


class AnalyticSphere:
    """Sphere represented as an IPNS (dual) sphere in CGA."""
    def __init__(self, center, radius, material=None):
        self.center = np.asarray(center, dtype=float)
        self.radius = float(radius)
        self.material = material or Material()
        # CGA IPNS sphere:  S = P_center - 0.5*r^2 * ni
        self.cga_sphere = cgaPointVec(self.center) - 0.5 * self.radius**2 * ni


class AnalyticPlane:
    """Infinite plane represented as an IPNS plane in CGA.
    Plane equation: n · x = d."""
    def __init__(self, normal, d, material=None):
        self.normal = np.asarray(normal, dtype=float)
        nn = np.linalg.norm(self.normal)
        self.normal = self.normal / nn
        self.d = float(d) / nn
        self.material = material or Material()
        # CGA IPNS plane:  pi = n + d*ni
        n_cga = self.normal[0]*e1 + self.normal[1]*e2 + self.normal[2]*e3
        self.cga_plane = n_cga + self.d * ni


class CheckerPlane(AnalyticPlane):
    """Checkerboard-patterned plane (for ground floor visualization)."""
    def __init__(self, normal, d, color1, color2, scale=1.0, **kw):
        super().__init__(normal, d, **kw)
        self.color1 = np.asarray(color1, dtype=float)
        self.color2 = np.asarray(color2, dtype=float)
        self.scale = scale


class PointLight:
    """Point light source. Position stored as CGA flat point (Section 23.4)."""
    def __init__(self, position, color=np.array([1.,1.,1.]), intensity=1.0):
        self.position = np.asarray(position, dtype=float)
        self.color = np.asarray(color, dtype=float)
        self.intensity = float(intensity)


# ---------- CGA transformations (Section 23.4) ----------

def translation_rotor(tx, ty, tz):
    """Translation versor:  T = exp(-0.5 * t ^ ni)  (book p.568)."""
    t = tx*e1 + ty*e2 + tz*e3
    return ONE - 0.5 * (t ^ ni)


def rotation_rotor(angle, axis_blade):
    """Rotation versor:  R = cos(a/2) - sin(a/2)*B   (Euclidean bivector B)."""
    return math.cos(angle/2) * ONE - math.sin(angle/2) * axis_blade


def scaling_rotor(s):
    """Uniform scaling versor:  S = exp(0.5*ln(s)*(no^ni))  (book p.568)."""
    return math.cosh(0.5*math.log(s)) * ONE + math.sinh(0.5*math.log(s)) * (no ^ ni)


def apply_versor(V, X):
    """Sandwich product  V X ~V  (book p.566)."""
    return V * X * ~V


# --- test transformations ---
P0 = cgaPoint(1, 0, 0)
T = translation_rotor(0, 0, 5)
P_translated = apply_versor(T, P0)
print('Translate (1,0,0) by (0,0,5):', down(P_translated))

R = rotation_rotor(math.pi/2, e1 ^ e2)  # 90° around z
P_rotated = apply_versor(R, P0)
print('Rotate (1,0,0) by 90° around z:', np.round(down(P_rotated), 6))

## Section 23.5 — Tracing the Rays

### 23.5.1 Ray Representation

Following the book (p.574), we represent a ray as a **(flat point, free vector)** pair:

```c++
class ray {
    flatPoint pos;        // position of the ray
    freeVector direction;  // direction of the ray
};
```

For efficiency we also keep the Euclidean origin and direction as numpy vectors.

In [ ]:
# Cell 5 — Ray class & spawning (Section 23.5.1 / 23.5.2)

@dataclass
class Ray:
    """Ray = flat point (position) + free vector (direction).
    We also cache Euclidean origin/dir for fast intersection."""
    origin: np.ndarray          # Euclidean [x,y,z]
    direction: np.ndarray       # Euclidean unit direction
    
    def __post_init__(self):
        self.direction = self.direction / np.linalg.norm(self.direction)


@dataclass
class HitRecord:
    """Information about a ray-object intersection."""
    t: float = float('inf')          # ray parameter
    point: np.ndarray = None         # Euclidean hit position
    normal: np.ndarray = None        # surface normal at hit
    material: Material = None
    hit: bool = False


# ----- Camera ray spawning (Section 23.5.2, book p.575) -----

class Camera:
    def __init__(self, position, look_at, up=np.array([0.,1.,0.]),
                 fov_deg=60.0, width=320, height=240):
        self.position = np.asarray(position, dtype=float)
        self.look_at = np.asarray(look_at, dtype=float)
        self.width = width
        self.height = height
        
        # Build orthonormal camera frame
        forward = self.look_at - self.position
        forward = forward / np.linalg.norm(forward)
        right = np.cross(forward, np.asarray(up, dtype=float))
        right = right / np.linalg.norm(right)
        cam_up = np.cross(right, forward)
        
        self.forward = forward
        self.right = right
        self.up = cam_up
        
        # FOV
        self.fov_width = 2.0 * math.tan(math.radians(fov_deg) / 2.0)
        self.fov_height = self.fov_width * height / width
    
    def spawn_ray(self, px, py):
        """Spawn a ray through pixel (px, py).  (book p.575)
        px, py in [0, width) x [0, height)."""
        # Normalized device coords [-0.5, 0.5]
        u = (px + 0.5) / self.width - 0.5
        v = (py + 0.5) / self.height - 0.5
        
        # Direction in world space (book: x*e1 + y*e2 + e3 mapped to camera frame)
        d = self.forward + u * self.fov_width * self.right + v * self.fov_height * self.up
        return Ray(origin=self.position.copy(), direction=d)


# --- test ---
cam = Camera(position=[0, 2, -5], look_at=[0, 0, 0], width=4, height=3)
r = cam.spawn_ray(2, 1.5)
print(f'Camera ray origin={r.origin}, dir={np.round(r.direction, 4)}')

## Section 23.5.3 — Ray–Object Intersection

### Bounding Sphere Test (book p.577)
```c++
line rayLine = ray.pos ^ (-no << ray.direction);
pointPair intersection = dual(rayLine) << boundingSphere;
if ((intersection << intersection) > 0) { /* hit */ }
```

We implement intersection for:
1. **Analytic sphere** — classical quadratic formula (fast) + CGA bounding check
2. **Analytic plane** — classical dot-product formula with checkerboard pattern support
3. **Triangle** — Moller-Trumbore (efficient, following the book's pragmatic approach)

In [ ]:
# Cell 6 — Intersection routines (Section 23.5.3)

EPSILON = 1e-7

def intersect_sphere(ray: Ray, sphere: AnalyticSphere) -> HitRecord:
    """Ray-sphere intersection.
    Uses CGA bounding check then classical quadratic for speed (book p.577)."""
    oc = ray.origin - sphere.center
    a = np.dot(ray.direction, ray.direction)      # = 1 for unit dir
    b = 2.0 * np.dot(oc, ray.direction)
    c = np.dot(oc, oc) - sphere.radius * sphere.radius
    disc = b * b - 4 * a * c
    if disc < 0:
        return HitRecord()
    sqrt_disc = math.sqrt(disc)
    t1 = (-b - sqrt_disc) / (2.0 * a)
    t2 = (-b + sqrt_disc) / (2.0 * a)
    t = t1 if t1 > EPSILON else t2
    if t < EPSILON:
        return HitRecord()
    pt = ray.origin + t * ray.direction
    n = (pt - sphere.center) / sphere.radius
    return HitRecord(t=t, point=pt, normal=n, material=sphere.material, hit=True)


def intersect_plane(ray: Ray, plane: AnalyticPlane) -> HitRecord:
    """Ray-plane intersection (signed distance from CGA, book p.578-579).
    Supports CheckerPlane pattern."""
    denom = np.dot(plane.normal, ray.direction)
    if abs(denom) < EPSILON:
        return HitRecord()       # parallel
    t = (plane.d - np.dot(plane.normal, ray.origin)) / denom
    if t < EPSILON:
        return HitRecord()
    pt = ray.origin + t * ray.direction
    n = plane.normal.copy()
    if denom > 0:                # flip normal to face the ray
        n = -n
    # Copy material so we can modify color per-hit for checkerboard
    mat = Material(
        color=plane.material.color.copy(),
        ambient=plane.material.ambient, diffuse=plane.material.diffuse,
        specular=plane.material.specular, shininess=plane.material.shininess,
        reflectivity=plane.material.reflectivity,
        transparency=plane.material.transparency, ior=plane.material.ior,
    )
    # Checkerboard pattern
    if isinstance(plane, CheckerPlane):
        x = pt[0] / plane.scale
        z = pt[2] / plane.scale
        if (int(math.floor(x)) + int(math.floor(z))) % 2 == 0:
            mat.color = plane.color1.copy()
        else:
            mat.color = plane.color2.copy()
    return HitRecord(t=t, point=pt, normal=n, material=mat, hit=True)


def intersect_triangle(ray: Ray, v0, v1, v2, face_normal):
    """Moller-Trumbore ray-triangle intersection.
    Returns (t, u, v) or None.  (pragmatic approach, book Section 23.5.3)"""
    edge1 = v1 - v0
    edge2 = v2 - v0
    h = np.cross(ray.direction, edge2)
    a = np.dot(edge1, h)
    if -EPSILON < a < EPSILON:
        return None
    f = 1.0 / a
    s = ray.origin - v0
    u = f * np.dot(s, h)
    if u < 0.0 or u > 1.0:
        return None
    q = np.cross(s, edge1)
    v = f * np.dot(ray.direction, q)
    if v < 0.0 or u + v > 1.0:
        return None
    t = f * np.dot(edge2, q)
    if t > EPSILON:
        return (t, u, v)
    return None


def intersect_mesh(ray: Ray, mesh: Mesh) -> HitRecord:
    """Ray-mesh intersection with bounding sphere pre-test (book p.577)."""
    # --- bounding sphere check ---
    oc = ray.origin - mesh.bounding_center
    b = 2.0 * np.dot(oc, ray.direction)
    c = np.dot(oc, oc) - mesh.bounding_radius**2
    disc = b*b - 4*c
    if disc < 0:
        return HitRecord()
    
    # --- test all faces ---
    best = HitRecord()
    for face in mesh.faces:
        v0 = mesh.vertices[face.idx[0]].pos
        v1 = mesh.vertices[face.idx[1]].pos
        v2 = mesh.vertices[face.idx[2]].pos
        result = intersect_triangle(ray, v0, v1, v2, face.normal)
        if result is not None:
            t, u, v = result
            if t < best.t:
                pt = ray.origin + t * ray.direction
                # Barycentric interpolation of vertex normals (book p.580, eq 11.19)
                w = 1.0 - u - v
                n0 = mesh.vertices[face.idx[0]].normal
                n1 = mesh.vertices[face.idx[1]].normal
                n2 = mesh.vertices[face.idx[2]].normal
                n_interp = w * n0 + u * n1 + v * n2
                nn = np.linalg.norm(n_interp)
                if nn > 1e-12:
                    n_interp = n_interp / nn
                mat = Material(color=mesh.color,
                               reflectivity=mesh.reflectivity,
                               transparency=mesh.transparency,
                               ior=mesh.ior)
                best = HitRecord(t=t, point=pt, normal=n_interp, material=mat, hit=True)
    return best


# --- CGA-based bounding sphere test (for demonstration, book p.577) ---
def cga_sphere_test(ray: Ray, sphere_dual_cga):
    """CGA bounding sphere check:  dual(L) ^ S  then check sign  (book p.577).
    Returns True if ray line intersects the sphere."""
    P1 = cgaPointVec(ray.origin)
    P2 = cgaPointVec(ray.origin + ray.direction)
    L_opns = P1 ^ P2 ^ ni                  # OPNS line
    L_ipns = L_opns * (-I5)                 # dual(L)
    pp_ipns = L_ipns ^ sphere_dual_cga      # IPNS point pair
    pp_opns = pp_ipns * (-I5)               # OPNS point pair
    sq = scalar_val(pp_opns | pp_opns)      # > 0 means real intersection
    return sq > 0


# --- quick tests ---
s = AnalyticSphere(center=[0,0,5], radius=1.0)
r = Ray(origin=np.array([0.,0.,0.]), direction=np.array([0.,0.,1.]))
hit = intersect_sphere(r, s)
print(f'Sphere hit: t={hit.t:.2f}, point={hit.point}, normal={hit.normal}')

p = AnalyticPlane(normal=[0,1,0], d=0.0)
r2 = Ray(origin=np.array([0.,2.,0.]), direction=np.array([0.,-1.,0.]))
hit2 = intersect_plane(r2, p)
print(f'Plane hit: t={hit2.t:.2f}, point={hit2.point}, normal={hit2.normal}')

# CGA bounding sphere test
print(f'CGA sphere test: {cga_sphere_test(r, s.cga_sphere)}')

## Section 23.5.4-23.5.5 — Reflection & Refraction

### Reflection (book p.579-580)
The reflected direction is computed by reflecting the free vector in the surface attitude plane:
```c++
// direction' = -(att ^ no) * direction * reverse(att ^ no)
```

### Refraction (book p.580, eq 23.1)
$$\mathbf{u}' = \left(\text{sign}(\mathbf{n}\cdot\mathbf{u})\sqrt{1-\eta^2+(\mathbf{n}\cdot\mathbf{u})^2\eta^2} - (\mathbf{n}\cdot\mathbf{u})\eta\right)\mathbf{n} + \eta\mathbf{u}$$

In [ ]:
# Cell 7 — Reflection & Refraction (Section 23.5.4 / 23.5.5)

def reflect_direction_cga(direction_vec, normal_vec):
    """CGA reflection of a free vector in a surface plane (book p.579-580).
    The surface attitude (free bivector) is dual(normal ^ ni).
    Reflection:  d' = -(att ^ no) * d_cga * reverse(att ^ no)
    We return the Euclidean reflected direction."""
    # Build CGA free vectors
    d_cga = (direction_vec[0]*e1 + direction_vec[1]*e2 + direction_vec[2]*e3) ^ ni
    n_cga = (normal_vec[0]*e1 + normal_vec[1]*e2 + normal_vec[2]*e3) ^ ni
    # Surface attitude: free bivector = dual of (normal ^ ni)
    att = cga_dual(n_cga)
    # Reflection plane through origin: att ^ no
    reflector = att ^ no
    # Sandwich: reflected = -reflector * d_cga * ~reflector
    reflected = -(reflector * d_cga * ~reflector)
    # Extract Euclidean direction from free vector
    rx = get_coeff(reflected, 'e14')
    ry = get_coeff(reflected, 'e24')
    rz = get_coeff(reflected, 'e34')
    r = np.array([rx, ry, rz])
    nn = np.linalg.norm(r)
    return r / nn if nn > 1e-12 else r


def reflect_direction(d, n):
    """Classical reflection:  d' = d - 2(d·n)n."""
    return d - 2.0 * np.dot(d, n) * n


def refract_direction(d, n, eta):
    """Refraction (Snell's law, book eq 23.1).
    d = incident direction (unit), n = surface normal (unit), eta = n1/n2.
    Returns refracted direction or None for total internal reflection."""
    cos_i = -np.dot(n, d)
    sin2_t = eta * eta * (1.0 - cos_i * cos_i)
    if sin2_t > 1.0:
        return None   # total internal reflection
    cos_t = math.sqrt(1.0 - sin2_t)
    return eta * d + (eta * cos_i - cos_t) * n


# --- test reflection ---
d_in = np.array([1., -1., 0.]); d_in = d_in / np.linalg.norm(d_in)
n_surf = np.array([0., 1., 0.])

d_ref_classic = reflect_direction(d_in, n_surf)
d_ref_cga = reflect_direction_cga(d_in, n_surf)
print(f'Reflection classic: {np.round(d_ref_classic, 4)}')
print(f'Reflection CGA:     {np.round(d_ref_cga, 4)}')
assert np.allclose(np.abs(d_ref_classic), np.abs(d_ref_cga), atol=1e-4), 'Reflection mismatch!'

# --- test refraction ---
d_refr = refract_direction(d_in, n_surf, 1.0/1.5)
print(f'Refraction (air->glass): {np.round(d_refr, 4)}')

## Section 23.6 — Shading

We use the **OpenGL fixed-function shading model** (ambient + diffuse + specular) with shadow checks, following Section 23.6 and [58] Chapter 5 as referenced in the book.

For each intersection point:
1. Spawn a **shadow ray** towards each light source (Section 23.5.2, p.576)
2. If unobstructed, compute **diffuse** $\max(0, \mathbf{n}\cdot\mathbf{l})$ and **specular** $(\mathbf{r}\cdot\mathbf{v})^{\alpha}$ contributions
3. Add **ambient** term
4. If material is reflective, spawn **reflection ray** and blend
5. If material is transparent, spawn **refraction ray** and blend

In [ ]:
# Cell 8 — Shading & complete ray tracer (Section 23.6)

class Scene:
    """The scene contains objects, lights, and the background."""
    def __init__(self):
        self.spheres: List[AnalyticSphere] = []
        self.planes: List[AnalyticPlane] = []
        self.meshes: List[Mesh] = []
        self.lights: List[PointLight] = []
        self.bg_color = np.array([0.2, 0.3, 0.5])  # sky
        self.ambient_color = np.array([1.0, 1.0, 1.0])
    
    def closest_hit(self, ray: Ray) -> HitRecord:
        """Find the closest intersection of a ray with the scene."""
        best = HitRecord()
        for sp in self.spheres:
            h = intersect_sphere(ray, sp)
            if h.hit and h.t < best.t:
                best = h
        for pl in self.planes:
            h = intersect_plane(ray, pl)
            if h.hit and h.t < best.t:
                best = h
        for ms in self.meshes:
            h = intersect_mesh(ray, ms)
            if h.hit and h.t < best.t:
                best = h
        return best
    
    def is_shadowed(self, point, light_pos):
        """Shadow ray test (Section 23.5.2, book p.576).
        Spawn a ray from surface point towards light; check obstruction."""
        to_light = light_pos - point
        dist = np.linalg.norm(to_light)
        if dist < EPSILON:
            return False
        shadow_ray = Ray(origin=point + EPSILON * (to_light / dist),
                         direction=to_light / dist)
        h = self.closest_hit(shadow_ray)
        return h.hit and h.t < dist


def shade(scene: Scene, ray: Ray, hit: HitRecord, depth: int) -> np.ndarray:
    """Compute color at hit point (Section 23.6)."""
    mat = hit.material
    color = mat.ambient * mat.color * scene.ambient_color
    
    V = -ray.direction  # view direction (towards camera)
    
    for light in scene.lights:
        # Shadow check (book p.576)
        if scene.is_shadowed(hit.point, light.position):
            continue
        
        L = light.position - hit.point
        dist = np.linalg.norm(L)
        L = L / dist
        
        # Diffuse (Lambertian)
        NdotL = max(0.0, np.dot(hit.normal, L))
        diffuse = mat.diffuse * NdotL * mat.color * light.color * light.intensity
        
        # Specular (Phong)
        R = reflect_direction(-L, hit.normal)
        RdotV = max(0.0, np.dot(R, V))
        specular = mat.specular * (RdotV ** mat.shininess) * light.color * light.intensity
        
        color = color + diffuse + specular
    
    # Reflection (Section 23.5.4, book p.579-580)
    if mat.reflectivity > 0 and depth > 0:
        refl_dir = reflect_direction(ray.direction, hit.normal)
        refl_ray = Ray(origin=hit.point + EPSILON * refl_dir, direction=refl_dir)
        refl_color = trace(scene, refl_ray, depth - 1)
        color = (1.0 - mat.reflectivity) * color + mat.reflectivity * refl_color
    
    # Refraction (Section 23.5.5, book p.580)
    if mat.transparency > 0 and depth > 0:
        # Determine if entering or exiting
        cos_i = np.dot(-ray.direction, hit.normal)
        if cos_i < 0:
            # Exiting: flip normal, invert eta
            n_refr = -hit.normal
            eta = mat.ior
        else:
            n_refr = hit.normal
            eta = 1.0 / mat.ior
        
        refr_dir = refract_direction(ray.direction, n_refr, eta)
        if refr_dir is not None:
            refr_ray = Ray(origin=hit.point + EPSILON * refr_dir, direction=refr_dir)
            refr_color = trace(scene, refr_ray, depth - 1)
            color = (1.0 - mat.transparency) * color + mat.transparency * refr_color
    
    return np.clip(color, 0.0, 1.0)


def trace(scene: Scene, ray: Ray, depth: int = 5) -> np.ndarray:
    """Trace a single ray into the scene (book Section 23.2)."""
    if depth <= 0:
        return scene.bg_color.copy()
    hit = scene.closest_hit(ray)
    if not hit.hit:
        return scene.bg_color.copy()
    return shade(scene, ray, hit, depth)


print('Shading & tracer engine ready.')

## Render Function

The main rendering loop spawns rays through each pixel (Section 23.5.2, book p.575) with optional multi-sampling for antialiasing.

In [ ]:
# Cell 9 — Renderer

def render(scene: Scene, camera: Camera, max_depth=5, samples=1):
    """Render the scene to an image (Section 23.5.2, book p.575-576)."""
    W, H = camera.width, camera.height
    image = np.zeros((H, W, 3))
    
    t0 = time.time()
    total = H * W
    
    for y in range(H):
        for x in range(W):
            color = np.zeros(3)
            for s in range(samples):
                # Antialiasing: small random offset per sample (book p.576)
                if samples > 1:
                    dx = np.random.rand() - 0.5
                    dy = np.random.rand() - 0.5
                else:
                    dx, dy = 0.0, 0.0
                ray = camera.spawn_ray(x + dx, y + dy)
                color += trace(scene, ray, max_depth)
            image[H - 1 - y, x] = color / samples  # flip y for image coords
        
        # Progress
        if (y + 1) % max(1, H // 10) == 0:
            elapsed = time.time() - t0
            pct = 100.0 * (y + 1) / H
            print(f'  {pct:5.1f}% ({elapsed:.1f}s)', end='\r')
    
    elapsed = time.time() - t0
    print(f'  Render complete: {W}x{H}, {samples} spp, {elapsed:.1f}s          ')
    return np.clip(image, 0.0, 1.0)


print('Renderer ready.')

## Build a Test Scene & Render

We build a scene inspired by the figures in Chapter 23 (Figure 23.1 / 23.2):
- A **reflective sphere** (like the geodesic sphere in the book's screenshot)
- A **refractive (glass) sphere**
- A **matte sphere**
- A **ground plane** with checkerboard pattern
- A triangulated **box** (mesh with CGA bounding sphere)
- Two point lights

In [ ]:
# Cell 10 — Build scene

scene = Scene()
scene.bg_color = np.array([0.1, 0.15, 0.3])  # dark sky

# --- Ground plane (y = -1) with checkerboard ---
scene.planes.append(CheckerPlane(
    normal=[0, 1, 0], d=-1.0,
    color1=np.array([0.9, 0.9, 0.9]),
    color2=np.array([0.2, 0.2, 0.2]),
    scale=2.0,
    material=Material(color=np.array([0.5, 0.5, 0.5]),
                      ambient=0.1, diffuse=0.6, specular=0.2,
                      shininess=16.0, reflectivity=0.25)
))

# --- Reflective sphere (center-left) ---
scene.spheres.append(AnalyticSphere(
    center=[-2.0, 0.5, 4.0], radius=1.5,
    material=Material(color=np.array([0.8, 0.8, 0.9]),
                      ambient=0.05, diffuse=0.3, specular=0.8,
                      shininess=128.0, reflectivity=0.7)
))

# --- Glass sphere (center-right) ---
scene.spheres.append(AnalyticSphere(
    center=[2.0, 0.0, 5.0], radius=1.0,
    material=Material(color=np.array([0.95, 0.95, 1.0]),
                      ambient=0.02, diffuse=0.1, specular=0.9,
                      shininess=256.0, reflectivity=0.1,
                      transparency=0.8, ior=1.5)
))

# --- Matte red sphere (far right) ---
scene.spheres.append(AnalyticSphere(
    center=[4.5, -0.2, 7.0], radius=0.8,
    material=Material(color=np.array([0.9, 0.2, 0.1]),
                      ambient=0.1, diffuse=0.8, specular=0.3,
                      shininess=32.0)
))

# --- Small golden sphere ---
scene.spheres.append(AnalyticSphere(
    center=[0.0, -0.5, 3.0], radius=0.5,
    material=Material(color=np.array([0.9, 0.7, 0.2]),
                      ambient=0.1, diffuse=0.6, specular=0.7,
                      shininess=64.0, reflectivity=0.3)
))

# --- Triangulated box (Section 23.3 mesh with CGA bounding sphere) ---
def make_box(cx, cy, cz, sx, sy, sz, color, reflectivity=0.0):
    """Create a box mesh centered at (cx,cy,cz) with half-extents (sx,sy,sz)."""
    corners = []
    for dx in [-sx, sx]:
        for dy in [-sy, sy]:
            for dz in [-sz, sz]:
                corners.append(np.array([cx+dx, cy+dy, cz+dz]))
    verts = []; faces = []
    face_defs = [
        ([0,4,6,2], [0,-1,0]), ([1,3,7,5], [0,1,0]),
        ([0,1,5,4], [0,0,-1]), ([2,6,7,3], [0,0,1]),
        ([0,2,3,1], [-1,0,0]), ([4,5,7,6], [1,0,0]),
    ]
    vi = 0
    for cidx, n in face_defs:
        n = np.array(n, dtype=float)
        for i in cidx:
            verts.append(Vertex(pos=corners[i].copy(), normal=n.copy()))
        faces.append((vi, vi+1, vi+2))
        faces.append((vi, vi+2, vi+3))
        vi += 4
    return Mesh(verts, faces, color=color, reflectivity=reflectivity)

box = make_box(-4.0, 0.0, 7.0, 0.8, 1.0, 0.8,
               color=np.array([0.3, 0.6, 0.9]), reflectivity=0.15)
scene.meshes.append(box)

# --- Lights ---
scene.lights.append(PointLight(position=np.array([-5., 8., -2.]),
                               color=np.array([1., 1., 0.95]), intensity=1.0))
scene.lights.append(PointLight(position=np.array([8., 6., 0.]),
                               color=np.array([0.8, 0.85, 1.0]), intensity=0.6))

print(f'Scene: {len(scene.spheres)} spheres, {len(scene.planes)} planes, '
      f'{len(scene.meshes)} meshes, {len(scene.lights)} lights')

In [ ]:
# Cell 11 — Render at low resolution first (quick validation)

cam = Camera(position=np.array([0., 2., -4.]),
             look_at=np.array([0., 0., 5.]),
             fov_deg=60, width=160, height=120)

img_low = render(scene, cam, max_depth=4, samples=1)

plt.figure(figsize=(10, 7.5))
plt.imshow(img_low)
plt.axis('off')
plt.title('CGA Ray Tracer — Low Resolution Preview (160x120)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Render at medium resolution

cam_hd = Camera(position=np.array([0., 2., -4.]),
                look_at=np.array([0., 0., 5.]),
                fov_deg=60, width=480, height=360)

img_med = render(scene, cam_hd, max_depth=5, samples=1)

plt.figure(figsize=(12, 9))
plt.imshow(img_med)
plt.axis('off')
plt.title('CGA Ray Tracer (Dorst Ch.23) — 480x360, depth=5')
plt.tight_layout()
plt.show()

## CGA Demonstration — Transforming the Scene

Following Section 23.4, we demonstrate CGA versors for scene manipulation:
- **Translation rotor** to move objects
- **Rotation rotor** to orbit the camera
- **Scaling rotor** to resize objects
- **CGA line-sphere intersection** (Section 23.5.3, book p.577)

All transformations use the sandwich product $V X \widetilde{V}$.

In [ ]:
# Cell 13 — CGA Transformation Demos (Section 23.4)

print('=== CGA Transformation Demonstrations (Section 23.4) ===')
print()

# 1. Translation rotor (book p.568)
T = translation_rotor(3, 0, 0)
P_orig = cgaPoint(0, 0, 5)
P_moved = apply_versor(T, P_orig)
print(f'Translation rotor T(3,0,0):')
print(f'  (0,0,5) -> {down(P_moved)}')

# 2. Rotation rotor (book p.569)
angle = math.pi / 4  # 45 degrees
R = rotation_rotor(angle, e1 ^ e3)  # rotation in xz-plane (around y)
P_rot = apply_versor(R, P_orig)
print(f'\nRotation rotor R(45deg, e1^e3):')
print(f'  (0,0,5) -> {np.round(down(P_rot), 4)}')

# 3. Scaling rotor (book p.568)
S = scaling_rotor(2.0)
P_scaled = apply_versor(S, P_orig)
print(f'\nScaling rotor S(2.0):')
print(f'  (0,0,5) -> {np.round(down(P_scaled), 4)}')

# 4. Composed: translate then rotate (book p.566: premultiply = global frame)
V = R * T  # first translate, then rotate (reading right to left)
P_comp = apply_versor(V, P_orig)
print(f'\nComposed T then R:')
print(f'  (0,0,5) -> {np.round(down(P_comp), 4)}')

# 5. CGA line-sphere intersection (book p.577)
print(f'\n=== CGA Line-Sphere Intersection (Section 23.5.3) ===')
ray_origin = cgaPoint(0, 0, -5)
ray_target = cgaPoint(0, 0, 5)
L_opns = ray_origin ^ ray_target ^ ni  # OPNS line
L_ipns = L_opns * (-I5)               # IPNS line (dual)

S_ipns = cgaPoint(0, 0, 3) - 0.5 * 1.0 * ni  # sphere at (0,0,3), r=1
pp_ipns = L_ipns ^ S_ipns  # IPNS point pair
pp_opns = pp_ipns * (-I5)  # OPNS point pair
pp_sq = scalar_val(pp_opns | pp_opns)
print(f'  Point pair squared norm: {pp_sq:.4f} (>0 = real intersection)')

if pp_sq > 0:
    # Extract the two intersection points (eq 14.13)
    n_val = math.sqrt(pp_sq)
    v1 = ni | pp_opns
    v2 = n_val * v1
    v3 = v1 | pp_opns
    scale_val = scalar_val(v3 | ni)
    scale = 1.0 / scale_val
    fp1 = (scale * ni) ^ (v3 + v2)
    fp2 = (scale * ni) ^ (v3 - v2)
    print(f'  Intersection 1: {down_flat(fp1)}')
    print(f'  Intersection 2: {down_flat(fp2)}')
    print(f'  Expected: (0, 0, 2) and (0, 0, 4)')

In [ ]:
# Cell 14 — Render from a different camera angle (demonstrating camera orbit, Section 23.4)

cam_orbit = Camera(
    position=np.array([6., 4., -2.]),
    look_at=np.array([0., 0., 5.]),
    fov_deg=55, width=480, height=360
)

img_orbit = render(scene, cam_orbit, max_depth=5, samples=1)

plt.figure(figsize=(12, 9))
plt.imshow(img_orbit)
plt.axis('off')
plt.title('CGA Ray Tracer — Orbited Camera View (Section 23.4)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 — High quality render with antialiasing

cam_final = Camera(
    position=np.array([0., 2.5, -5.]),
    look_at=np.array([0., 0., 5.]),
    fov_deg=58, width=640, height=480
)

print('Rendering final image at 640x480 with 4x antialiasing...')
img_final = render(scene, cam_final, max_depth=5, samples=4)

plt.figure(figsize=(14, 10.5))
plt.imshow(img_final)
plt.axis('off')
plt.title('CGA Ray Tracer (Dorst et al. Ch.23) — 640x480, 4 spp, depth=5')
plt.tight_layout()
plt.show()

# Save to file
img_pil = Image.fromarray((img_final * 255).astype(np.uint8))
img_pil.save('GA-raytracing-output.png')
print('Saved to GA-raytracing-output.png')

## Section 23.7 — Evaluation

Following the chapter's conclusion:

- **CGA representations** were used for all geometric primitives: points (`cgaPoint`), planes (IPNS), spheres (IPNS), lines (OPNS), and bounding spheres.
- **CGA versors** (rotors) were used for translations, rotations, and scaling via the sandwich product.
- **CGA intersection** was demonstrated for the bounding sphere test using `dual(L) ^ S` in IPNS representation, and point-pair extraction via eq. 14.13.
- For **ray-triangle** and **ray-analytic** intersection, we followed the book's pragmatic approach of using efficient coordinate-level computations while keeping the CGA framework for representation and transformation.
- **Reflection** was implemented both via the CGA sandwich product (Section 23.5.4) and the classical formula, with agreement validated.
- **Refraction** follows equation 23.1.
- **Shading** uses the OpenGL fixed-function model (ambient + diffuse + specular) with shadow rays.

As Dorst notes: *"Most interactive modeling transformations have been reduced to simple one-liners, direct conformal model formulas... This is a part of the code where the conformal model shows off some of its power."*